# Wine Clustering Comparison

## Project Overview
This notebook compares three unsupervised learning approaches on the Wine dataset:

- **KMeans**
- **DBSCAN**
- **Agglomerative Clustering**

The goal is to understand how different clustering methods behave on the same dataset after preprocessing and dimensionality reduction.

## What this project shows
- data preprocessing for unsupervised learning
- feature scaling
- cluster selection with the elbow method and silhouette score
- PCA-based visualization
- comparison of clustering algorithms
- interpretation of cluster structure


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

try:
    from scipy.cluster.hierarchy import linkage, dendrogram
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)

## 1. Load the Dataset

In [ ]:
data = load_wine(as_frame=True)
X = data.data.copy()
y_true = data.target.copy()

print("Dataset shape:", X.shape)
print("Number of features:", X.shape[1])
X.head()

## 2. Basic Dataset Information

The Wine dataset contains chemical analysis features for wines from three cultivars.
Since clustering is unsupervised, the target labels are **not** used for training, but they can help us interpret the structure later.


In [ ]:
summary_df = pd.DataFrame({
    "feature": X.columns,
    "mean": X.mean().round(2).values,
    "std": X.std().round(2).values,
    "min": X.min().round(2).values,
    "max": X.max().round(2).values
})
summary_df.head(10)

## 3. Standardize Features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled data shape:", X_scaled.shape)
print("Approximate mean after scaling:", np.round(X_scaled.mean(axis=0)[:5], 4))
print("Approximate std after scaling:", np.round(X_scaled.std(axis=0)[:5], 4))

## 4. KMeans: Elbow Method

In [ ]:
inertias = []
ks = range(2, 11)

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(dpi=160)
plt.plot(list(ks), inertias, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for KMeans")
plt.tight_layout()
plt.show()

## 5. KMeans: Silhouette Score

The silhouette score helps measure how well-separated the clusters are.
A higher score usually indicates better-defined clusters.


In [ ]:
sil_scores = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

silhouette_df = pd.DataFrame({
    "k": list(ks),
    "silhouette_score": np.round(sil_scores, 4)
})

plt.figure(dpi=160)
plt.plot(list(ks), sil_scores, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score vs Number of Clusters")
plt.tight_layout()
plt.show()

best_k = list(ks)[int(np.argmax(sil_scores))]
print("Best k by silhouette:", best_k)
silhouette_df

## 6. Fit KMeans with the Selected Number of Clusters

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

kmeans_counts = pd.Series(kmeans_labels).value_counts().sort_index()
print("Cluster counts:")
print(kmeans_counts)

## 7. Cluster Profile for KMeans

In [ ]:
df_kmeans = X.copy()
df_kmeans["cluster"] = kmeans_labels

cluster_profile = df_kmeans.groupby("cluster").mean().round(2)
cluster_profile

In [ ]:
feature_spread = (cluster_profile.max() - cluster_profile.min()).sort_values(ascending=False)
feature_spread.head(10).to_frame("difference_between_cluster_max_and_min")

## 8. PCA for 2D Visualization

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
print("Explained variance ratio:", np.round(explained, 4))
print("Total explained by first 2 PCs:", np.round(explained.sum(), 4))

## 9. KMeans Clusters in PCA Space

In [ ]:
plt.figure(dpi=180)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("KMeans Clusters (PCA Projection)")
plt.tight_layout()
plt.show()

## 10. Agglomerative Clustering

In [ ]:
agg = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
agg_labels = agg.fit_predict(X_scaled)

agg_sil = silhouette_score(X_scaled, agg_labels)
print("Agglomerative silhouette score:", round(agg_sil, 4))
pd.Series(agg_labels).value_counts().sort_index().to_frame("count")

In [ ]:
plt.figure(dpi=180)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=agg_labels)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Agglomerative Clustering (PCA Projection)")
plt.tight_layout()
plt.show()

## 11. Optional Dendrogram

In [ ]:
if SCIPY_AVAILABLE:
    Z = linkage(X_scaled, method="ward")
    plt.figure(figsize=(10, 4), dpi=160)
    dendrogram(Z, truncate_mode="lastp", p=20)
    plt.title("Truncated Dendrogram (Ward Linkage)")
    plt.xlabel("Cluster Index")
    plt.ylabel("Distance")
    plt.tight_layout()
    plt.show()
else:
    print("Scipy hierarchy tools are not available in this runtime.")

## 12. DBSCAN

In [ ]:
dbscan = DBSCAN(eps=1.2, min_samples=6)
db_labels = dbscan.fit_predict(X_scaled)

unique, counts = np.unique(db_labels, return_counts=True)
db_info = pd.DataFrame({"label": unique, "count": counts})
db_info

In [ ]:
n_noise = int((db_labels == -1).sum())
n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
print("Estimated number of DBSCAN clusters:", n_clusters_db)
print("Number of noise points:", n_noise)

In [ ]:
plt.figure(dpi=180)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=db_labels)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("DBSCAN Clusters (PCA Projection)")
plt.tight_layout()
plt.show()

## 13. Sensitivity of DBSCAN to `eps`

In [ ]:
eps_results = []

for eps in [0.8, 1.0, 1.2, 1.5, 1.8]:
    db = DBSCAN(eps=eps, min_samples=6)
    labels = db.fit_predict(X_scaled)
    n_noise = int((labels == -1).sum())
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    eps_results.append({
        "eps": eps,
        "clusters_found": n_clusters,
        "noise_points": n_noise
    })

pd.DataFrame(eps_results)

## 14. Comparison Summary

In [ ]:
comparison_rows = []

kmeans_sil = silhouette_score(X_scaled, kmeans_labels)
comparison_rows.append({
    "method": "KMeans",
    "clusters_found": len(np.unique(kmeans_labels)),
    "noise_points": 0,
    "silhouette_score": round(kmeans_sil, 4)
})

comparison_rows.append({
    "method": "Agglomerative",
    "clusters_found": len(np.unique(agg_labels)),
    "noise_points": 0,
    "silhouette_score": round(agg_sil, 4)
})

db_unique = set(db_labels)
db_clusters_found = len(db_unique) - (1 if -1 in db_unique else 0)
if db_clusters_found >= 2:
    db_sil = round(silhouette_score(X_scaled, db_labels), 4)
else:
    db_sil = np.nan

comparison_rows.append({
    "method": "DBSCAN",
    "clusters_found": db_clusters_found,
    "noise_points": int((db_labels == -1).sum()),
    "silhouette_score": db_sil
})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

## 15. Final Interpretation

In [ ]:
print("""Key Insights:

1. KMeans produced clear and well-separated clusters on the standardized wine dataset.
2. Agglomerative clustering showed behavior similar to KMeans and also performed well.
3. DBSCAN was more sensitive to parameter choices and treated some observations as noise.
4. PCA made it easier to visualize clustering structure in two dimensions.
5. This project shows why comparing multiple clustering algorithms is important in unsupervised learning.
""")

## Conclusion

This portfolio project demonstrates a complete clustering workflow:
- preprocessing the data
- selecting model settings
- visualizing clusters
- comparing algorithms
- interpreting the results

For GitHub and LinkedIn, the most useful visuals from this notebook are:
1. the **Elbow Method** plot
2. the **Silhouette Score vs k** plot
3. the **KMeans PCA projection**
4. the **Agglomerative PCA projection**
5. the **DBSCAN PCA projection**
6. the **comparison summary table**
